In [1]:
topic_name = "COVID"

CALCULATE TOP DOMAINS PER MODULARITY CLASS

In [ ]:
import pandas as pd
from pathlib import Path

path = Path('/Users/rubenvandijkhuizen/Library/CloudStorage/OneDrive-Personal/Studie/Master/Thesis/Gephi/Topic_COVID.csv')
df = pd.read_csv(path)

# identify likely columns
cols = {c.lower(): c for c in df.columns}
mod_col = cols.get('modularity class') or cols.get('modularity_class') or cols.get('modularity')
wd_col = cols.get('weighted in-degree') or cols.get('weighted_in-degree') or cols.get('weighted in degree') or cols.get('weighted indegree')
id_col = cols.get('id')

a = df[[mod_col, id_col, wd_col]].copy()
a[wd_col] = pd.to_numeric(a[wd_col], errors='coerce').fillna(0)
a[mod_col] = a[mod_col].astype(str)

def top10(g):
    out = g.groupby(id_col, as_index=False)[wd_col].sum().sort_values([wd_col, id_col], ascending=[False, True]).head(10)
    out.insert(0, mod_col, g.name)
    return out

res = a.groupby(mod_col, group_keys=False).apply(top10).reset_index(drop=True)
res = res.rename(columns={mod_col: 'modularity_class', id_col: 'domain_name', wd_col: 'weighted_in_degree'})

out_dir = Path('output')
out_dir.mkdir(exist_ok=True)
res.to_csv(out_dir / f'{topic_name}_top10_domain_names_by_modularity_class.csv', index=False)



/var/folders/08/nkwn4hx57xq3hlr21l21gykm0000gn/T/ipykernel_42684/4293465885.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  res = a.groupby(mod_col, group_keys=False).apply(top10).reset_index(drop=True)


,modularity_class,domain_name,weighted_in_degree
0,0,Website@www.ad.nl,867
1,0,Website@www.telegraaf.nl,773
2,0,Website@www.nu.nl,643
3,0,Website@nos.nl,549
4,0,Website@www.rt.com,338
5,0,Website@www.rtlnieuws.nl,338
6,0,Website@f7td5.app.goo.gl,232
7,0,Website@www.rtl.nl,164
8,0,Website@www.volkskrant.nl,146
9,0,Website@www.trouw.nl,97


CALCULATE TOP10 & TOP20 SHARES 

In [ ]:
ranked=df.sort_values('weighted indegree', ascending=False).reset_index(drop=True)

ranked['rank']=ranked.index+1

total_wd=ranked['weighted indegree'].sum()

share10 = ranked.head(10)['weighted indegree'].sum()/total_wd
share20 = ranked.head(20)['weighted indegree'].sum()/total_wd

out=pd.DataFrame({'metric':['top_10_share_of_total_weighted_indegree','top_20_share_of_total_weighted_indegree','total_weighted_indegree'], 'value':[share10,share20,total_wd]})

out.to_csv(f'output/{topic_name}_top_shares.csv', index=False)

CALCULATE MODULARITY CLASS STATS

In [4]:
# modularity composition
mod=df.groupby('modularity_class').agg(nodes=('Id','size'), total_weighted_indegree=('weighted indegree','sum'), median_indegree=('indegree','median')).sort_values('nodes', ascending=False)
mod.to_csv(f'output/{topic_name}modularity_summary.csv')